This notebook fetch precomputed 3D protein structures from AlpahfoldDB using uniprot IDs as input.

**Input format ** (.txt file) contains IDS as follows,

ID 1

ID 2

ID 3

In [ ]:
import requests
import os
import shutil
from google.colab import files
from tqdm.notebook import tqdm  # Specialized progress bar for Colab/Jupyter

def download_alphafold_cif():
    print("Please upload your .txt file containing UNIPROT IDs:")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return

    input_filename = list(uploaded.keys())[0]
    content = uploaded[input_filename].decode("utf-8")
    ids = [line.strip() for line in content.splitlines() if line.strip()]

    folder_name = "alphafold_data"
    if os.path.exists(folder_name):
        shutil.rmtree(folder_name)
    os.makedirs(folder_name)

    base_url = "https://www.ebi.ac.uk/Tools/dbfetch/dbfetch"
    downloaded_count = 0
    errors = []

    print(f"\nProcessing {len(ids)} IDs...")
    for entry_id in tqdm(ids, desc="Downloading structures"):
        params = {'db': 'alphafolddb', 'id': entry_id, 'format': 'mmcif'}
        try:
            response = requests.get(base_url, params=params)
            if response.status_code == 200:
                file_path = os.path.join(folder_name, f"{entry_id}.cif")
                with open(file_path, 'wb') as f:
                    f.write(response.content)
                downloaded_count += 1
            else:
                errors.append(f"{entry_id} (Status {response.status_code})")
        except Exception as e:
            errors.append(f"{entry_id} (Error: {e})")
    if downloaded_count > 0:
        zip_name = "alphafold_structures"
        shutil.make_archive(zip_name, 'zip', folder_name)
        print(f"\n Done. {downloaded_count} files downloaded.")
        if errors:
            print(f"Failed to download: {', '.join(errors)}")

        files.download(f"{zip_name}.zip")
    else:
        print("\n No files were downloaded.")

download_alphafold_cif()

Please upload your .txt file containing UNIPROT IDs:


Saving unique_in_input_only.txt to unique_in_input_only (1).txt

Processing 114 IDs...



 Done. 114 files downloaded.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>